# Base Model Inference Test — Qwen2.5-3B

This notebook evaluates the inference capability of the **Qwen2.5-3B** base model before fine-tuning.

**Contents:**
1. Environment check (GPU, CUDA)
2. Load model & tokenizer
3. Text generation test
4. Meeting summary prompt test
5. Timing & memory measurement

## 1. Environment Check

In [1]:
import sys
import torch

print(f"Python : {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA   : {torch.version.cuda}")
print(f"GPU    : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU available'}")
print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "")

Python : 3.10.11 (tags/v3.10.11:7d4cc5a, Apr  5 2023, 00:38:17) [MSC v.1929 64 bit (AMD64)]
PyTorch: 2.10.0+cpu
CUDA   : None
GPU    : No GPU available



In [1]:
# Optional: Hugging Face login
from huggingface_hub import login
from getpass import getpass

# hf_token = getpass("Enter your Hugging Face token here: ")
# login(token=hf_token)


c:\Users\ezycloudx-admin\Downloads\qwen2.5-3b-meeting-summarization\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Load Model & Tokenizer

In [4]:
import sys
import os

# Add project root to sys.path for module imports
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from modules.model_loader import load_model_and_tokenizer

In [5]:
# Load model with 4-bit quantization to reduce VRAM usage
# Set use_4bit=False to load in full precision
model, tokenizer = load_model_and_tokenizer(
    use_4bit=True,
    torch_dtype="bfloat16",
    device_map="auto",
)

print(f"Model device : {model.device}")
print(f"Model dtype  : {model.dtype}")
print(f"Vocab size   : {tokenizer.vocab_size:,}")
print(f"Pad token    : {tokenizer.pad_token} (id={tokenizer.pad_token_id})")

c:\Users\ezycloudx-admin\Downloads\qwen2.5-3b-meeting-summarization\venv\lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ezycloudx-admin\Downloads\qwen2.5-3b-meeting-summarization\models\models--Qwen--Qwen2.5-3B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
`torch_dtype` is deprecated! Use `dtype` in

Model device : cuda:0
Model dtype  : torch.bfloat16
Vocab size   : 151,643
Pad token    : <|endoftext|> (id=151643)


## 3. Text Generation Helper Function

In [ ]:
import time

def generate_text(
    prompt: str,
    max_new_tokens: int = 256,
    temperature: float = 0.7,
    top_p: float = 0.9,
    top_k: int = 50,
    do_sample: bool = True,
    repetition_penalty: float = 1.1,
) -> str:
    """Generate text from a prompt and measure elapsed time."""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]

    start = time.perf_counter()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            do_sample=do_sample,
            repetition_penalty=repetition_penalty,
            pad_token_id=tokenizer.pad_token_id,
        )

    elapsed = time.perf_counter() - start
    generated_tokens = outputs.shape[1] - input_len

    result = tokenizer.decode(outputs[0], skip_special_tokens=True)

    print(f" Time  : {elapsed:.2f}s")
    print(f" Tokens generated : {generated_tokens}")
    print(f" Speed     : {generated_tokens / elapsed:.1f} tokens/s")
    print("-" * 60)

    return result

## 4. Basic Text Generation Test

In [13]:
prompt_1 = "Artificial intelligence is"

result = generate_text(prompt_1, max_new_tokens=128)
print(result)

⏱  Thời gian  : 7.22s
📝 Tokens sinh : 128
⚡ Tốc độ     : 17.7 tokens/s
------------------------------------------------------------
Artificial intelligence is the science of making computers think and behave like humans. It includes two types: narrow AI (also known as weak AI) which can only perform a specific task, and general AI, also known as strong AI, which can perform any intellectual task that a human being can.
There are three main approaches to AI:
- Symbolic Approach – using algorithms
- Neural Networks – using an artificial neural network (ANN)
- Connectionism – using deep learning
This post will focus on symbolic approach, where AI uses a set of rules to make decisions based on input data.
The Rules of Logic
Logic is one of the most ancient sciences in


In [14]:
prompt_2 = "The key benefits of using large language models in business are:"

result = generate_text(prompt_2, max_new_tokens=200)
print(result)

⏱  Thời gian  : 8.01s
📝 Tokens sinh : 140
⚡ Tốc độ     : 17.5 tokens/s
------------------------------------------------------------
The key benefits of using large language models in business are: 
1. **Improved Efficiency**: Large language models can process and analyze vast amounts of data quickly, leading to faster decision-making processes.
2. **Enhanced Accuracy**: They provide more accurate predictions and insights compared to traditional methods, which helps in making better-informed decisions.
3. **Cost Savings**: By automating repetitive tasks such as customer service interactions or content creation, businesses save on labor costs while improving quality.

### Key Benefits
- **Efficiency**: Process large volumes of data swiftly, enabling quick response times and informed decisions.
- **Accuracy**: Provide precise predictions and insights, enhancing overall performance.
- **Cost Savings**: Automate routine tasks, reducing human intervention and associated expenses.


## 5. Meeting Summary Prompt Test

This is the core task of the project — evaluating how the base model handles meeting summarization prompts **before fine-tuning**.

In [15]:
meeting_transcript = """Meeting Transcript:
John: Good morning everyone. Let's start with the Q3 report.
Sarah: Revenue is up 15% compared to last quarter. We hit $2.3 million.
John: Great news. What about customer acquisition?
Mike: We onboarded 340 new customers. Churn rate dropped to 3.2%.
Sarah: Marketing spent $180K this quarter, mainly on digital campaigns.
John: ROI looks solid. Any concerns?
Mike: We need to invest more in customer support. Response times are increasing.
John: Agreed. Let's allocate budget for that next quarter.
Sarah: I'll prepare a proposal by Friday.
John: Perfect. Meeting adjourned.

Summary of this meeting:"""

result = generate_text(meeting_transcript, max_new_tokens=256)
print(result)

⏱  Thời gian  : 4.14s
📝 Tokens sinh : 71
⚡ Tốc độ     : 17.2 tokens/s
------------------------------------------------------------
Meeting Transcript:
John: Good morning everyone. Let's start with the Q3 report.
Sarah: Revenue is up 15% compared to last quarter. We hit $2.3 million.
John: Great news. What about customer acquisition?
Mike: We onboarded 340 new customers. Churn rate dropped to 3.2%.
Sarah: Marketing spent $180K this quarter, mainly on digital campaigns.
John: ROI looks solid. Any concerns?
Mike: We need to invest more in customer support. Response times are increasing.
John: Agreed. Let's allocate budget for that next quarter.
Sarah: I'll prepare a proposal by Friday.
John: Perfect. Meeting adjourned.

Summary of this meeting: John and Sarah presented the Q3 revenue figures which were slightly above their expectations. They also discussed the performance of customer acquisition and churn rate. Mike expressed his concern over the increase in response times and suggested a

In [22]:
# Test with a Vietnamese meeting prompt
meeting_vn = """
Bạn là trợ lý AI chuyên tóm tắt cuộc họp tiếng việt. Hãy tóm tắt đầy đủ ý, ngắn gọn, súc
tích.
Hãy tóm tắt cuộc họp tuân theo định dạng sau đây:
# Triển khai hệ thống tóm tắt cuộc họp tự động
## I. Nội dung chính
### 1. Mục tiêu cuộc họp
### 2. Các vấn đề đã thảo luận
### 3. Kết luận và quyết định
## II. Danh sách công việc cần làm

| Công việc | Người phụ trách | Hạn chót |
| :--- | :--- | :--- |
|  |  |  |
|  |  |  |
|  |  |  |
|  |  |  |
|  |  |  |

Transcripts:
[08:30] Chào anh chị, sáng nay chúng ta ngồi lại để xem xét tình hình thị trường cho đợt ra mắt dòng sản phẩm gia dụng thông minh sắp tới.
[08:32] Theo báo cáo mới nhất từ bộ phận nghiên cứu, sức mua của người tiêu dùng đang có dấu hiệu hồi phục nhưng họ lại khắt khe hơn về giá.
[08:34] Đúng vậy, phân khúc khách hàng mục tiêu của mình là tầng lớp trung lưu, họ quan tâm nhiều đến tính năng tiết kiệm điện và thiết kế tối giản.
[08:36] Tôi thấy các đối thủ cạnh tranh như Brand X đang đẩy mạnh khuyến mãi rất sâu, giảm tới 20% cho các gói combo.
[08:38] Nếu mình cũng chạy đua giảm giá thì biên lợi nhuận sẽ mỏng lắm, không ổn đâu.
[08:40] Thay vì giảm giá trực tiếp, mình nên tập trung vào giá trị gia tăng, ví dụ như tăng thời gian bảo hành lên 3 năm hoặc tặng kèm bộ lọc thay thế.
[08:42] Ý tưởng hay đó, khách hàng sẽ cảm thấy an tâm hơn khi mua đồ công nghệ mới.
[08:44] Về kênh phân phối, hiện tại hệ thống đại lý đang yêu cầu chiết khấu cao hơn để hỗ trợ trưng bày.
[08:46] Chúng ta có thể cân nhắc hỗ trợ họ chi phí marketing tại điểm bán thay vì tăng chiết khấu trực tiếp.
[08:48] Ngoài ra, cần đẩy mạnh cả kênh thương mại điện tử nữa, livestream đang là xu hướng rất tốt để giới thiệu tính năng sản phẩm.
[08:50] Tôi muốn đội Marketing chuẩn bị kịch bản livestream chi tiết, tập trung vào việc giải quyết các nỗi lo của khách hàng về độ bền.
[08:52] Bộ phận kho cũng báo là linh kiện nhập khẩu đang về chậm do ảnh hưởng vận tải biển, cần có kế hoạch dự phòng.
[08:54] Anh Nam xem lại lịch trình nhập hàng, nếu cần thì chuyển sang đường hàng không cho các linh kiện quan trọng để kịp ngày ra mắt.
[08:56] Vâng, tôi sẽ làm việc lại với bên logistics ngay chiều nay.
[08:58] Chốt lại là mình vẫn giữ ngày ra mắt dự kiến là 15 tháng tới nhé.
[09:00] Mọi người cố gắng bám sát timeline, có vấn đề gì phát sinh thì báo ngay cho tôi.
"""

result = generate_text(meeting_vn, max_new_tokens=256)
print(result)

⏱  Thời gian  : 14.70s
📝 Tokens sinh : 256
⚡ Tốc độ     : 17.4 tokens/s
------------------------------------------------------------

Bạn là trợ lý AI chuyên tóm tắt cuộc họp tiếng việt. Hãy tóm tắt đầy đủ ý, ngắn gọn, súc
tích.
Hãy tóm tắt cuộc họp tuân theo định dạng sau đây:
# Triển khai hệ thống tóm tắt cuộc họp tự động
## I. Nội dung chính
### 1. Mục tiêu cuộc họp
### 2. Các vấn đề đã thảo luận
### 3. Kết luận và quyết định
## II. Danh sách công việc cần làm

| Công việc | Người phụ trách | Hạn chót |
| :--- | :--- | :--- |
|  |  |  |
|  |  |  |
|  |  |  |
|  |  |  |
|  |  |  |

Transcripts:
[08:30] Chào anh chị, sáng nay chúng ta ngồi lại để xem xét tình hình thị trường cho đợt ra mắt dòng sản phẩm gia dụng thông minh sắp tới.
[08:32] Theo báo cáo mới nhất từ bộ phận nghiên cứu, sức mua của người tiêu dùng đang có dấu hiệu hồi phục nhưng họ lại khắt khe hơn về giá.
[08:34] Đúng vậy, phân khúc khách hàng mục tiêu của mình là tầng lớp trung lưu, họ quan tâm nhiều đến tính năng tiết

## 6. VRAM Usage

In [24]:
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved  = torch.cuda.memory_reserved() / 1e9
    total     = torch.cuda.get_device_properties(0).total_memory / 1e9

    print(f"VRAM Allocated : {allocated:.2f} GB")
    print(f"VRAM Reserved  : {reserved:.2f} GB")
    print(f"VRAM Total     : {total:.1f} GB")
    print(f"VRAM Free      : {total - reserved:.2f} GB")
else:
    print("No GPU detected — running on CPU.")

VRAM Allocated : 2.08 GB
VRAM Reserved  : 6.16 GB
VRAM Total     : 17.1 GB
VRAM Free      : 10.95 GB


## 7. Greedy vs Sampling Comparison

In [25]:
test_prompt = "The most important thing in a meeting is"

print("=" * 60)
print("GREEDY (do_sample=False)")
print("=" * 60)
greedy = generate_text(test_prompt, max_new_tokens=100, do_sample=False)
print(greedy)

print("\n")

print("=" * 60)
print("SAMPLING (temperature=0.7, top_p=0.9)")
print("=" * 60)
sampled = generate_text(test_prompt, max_new_tokens=100, do_sample=True, temperature=0.7)
print(sampled)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


GREEDY (do_sample=False)
⏱  Thời gian  : 5.95s
📝 Tokens sinh : 100
⚡ Tốc độ     : 16.8 tokens/s
------------------------------------------------------------
The most important thing in a meeting is to make sure that everyone has the same understanding of what you are trying to achieve. This means that you need to be clear about your objectives and have a plan for how you will get there.
A good way to start off a meeting is by setting an agenda, which should include all the topics that you want to cover during the session. It’s also helpful if you can give each item on the list a time limit so that people don’t spend too long discussing one topic at the expense of others


SAMPLING (temperature=0.7, top_p=0.9)
⏱  Thời gian  : 5.62s
📝 Tokens sinh : 100
⚡ Tốc độ     : 17.8 tokens/s
------------------------------------------------------------
The most important thing in a meeting is not to leave the people who are there feeling frustrated and uninvolved. It’s easy to fall into the trap of 

## 8. Free Memory

In [26]:
# Run this cell to release VRAM
import gc

del model
del tokenizer
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"VRAM after release: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

print("Memory released.")

VRAM sau giải phóng: 0.01 GB
Đã giải phóng bộ nhớ.
